# Phase 5: Feature Engineering

## Automated Data Quality & ETL Framework

This notebook creates analytical features from the original Online Retail dataset.  
The objective of this phase is to convert raw transactional columns into business-ready columns for analysis, reporting, and downstream ETL use.

### Main outputs

- `outputs/feature_engineered_online_retail.csv`
- `reports/feature_engineering/feature_engineering_summary.csv`

### Key feature groups

- Date and time features
- Revenue features
- Cancellation indicators
- Customer behavior features
- Region grouping
- Basic product category classification


## 1. Import Required Libraries

`pandas` is used for dataframe operations.  
`numpy` is used for conditional feature creation.  
`os` is used to create output folders if they do not already exist.


In [ ]:
import os

import numpy as np
import pandas as pd

## 2. Define Project Paths

Update `project_path` only if your main project folder location changes.

Expected project folder:

```text
Automated-Data-Quality-ETL-Framework
├── data
│   └── raw
│       └── Online Retail.xlsx
├── outputs
└── reports
```


In [ ]:
# Main project folder
project_path = r"C:\Users\chatu\OneDrive\Desktop\Automated-Data-Quality-ETL-Framework"

# Input file path - original raw dataset
input_file_path = os.path.join(project_path, "data", "raw", "Online Retail.xlsx")

# Output folders
outputs_folder = os.path.join(project_path, "outputs")
feature_report_folder = os.path.join(project_path, "reports", "feature_engineering")

# Create folders if they do not already exist
os.makedirs(outputs_folder, exist_ok=True)
os.makedirs(feature_report_folder, exist_ok=True)

# Output file paths
feature_output_path = os.path.join(outputs_folder, "feature_engineered_online_retail.csv")
feature_summary_path = os.path.join(feature_report_folder, "feature_engineering_summary.csv")

print("Project paths configured successfully.")

## 3. Load the Original Dataset

The feature engineering phase uses the original raw Excel file, not the cleaned CSV output.  
This keeps the pipeline traceable from raw source data to engineered output.


In [ ]:
# Load the original Online Retail dataset
df = pd.read_excel(input_file_path)

# Display dataset size
print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

## 4. Create a Working Copy

The raw dataframe is kept unchanged.  
All engineered columns are added to `feature_df`.


In [ ]:
feature_df = df.copy()

print("Working copy created successfully.")

## 5. Basic Column Checks

Before creating features, check whether the required source columns are available.


In [ ]:
required_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

missing_columns = [col for col in required_columns if col not in feature_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
else:
    print("All required columns are available.")

## 6. Convert Date Column

`InvoiceDate` must be converted to datetime format before extracting year, month, weekday, and hour features.


In [ ]:
feature_df["InvoiceDate"] = pd.to_datetime(feature_df["InvoiceDate"], errors="coerce")

print("InvoiceDate converted to datetime format.")

## 7. Create Date and Time Features

These features support monthly trend analysis, weekday analysis, quarter analysis, and peak purchase hour analysis.


In [ ]:
feature_df["Invoice_Year"] = feature_df["InvoiceDate"].dt.year
feature_df["Invoice_Month"] = feature_df["InvoiceDate"].dt.month
feature_df["Month_Name"] = feature_df["InvoiceDate"].dt.month_name()
feature_df["Quarter"] = feature_df["InvoiceDate"].dt.quarter
feature_df["Invoice_Day"] = feature_df["InvoiceDate"].dt.day
feature_df["Weekday"] = feature_df["InvoiceDate"].dt.day_name()
feature_df["Hour"] = feature_df["InvoiceDate"].dt.hour

print("Date and time features created successfully.")

## 8. Create Revenue Feature

`Total_Amount` is calculated as:

```text
Quantity × UnitPrice
```

This becomes the main revenue feature for sales analysis.


In [ ]:
feature_df["Total_Amount"] = feature_df["Quantity"] * feature_df["UnitPrice"]

print("Revenue feature created successfully.")

## 9. Create Cancellation Flag

In the Online Retail dataset, cancelled invoice numbers usually start with `C`.

The `Is_Cancelled` column helps separate normal sales transactions from cancelled transactions.


In [ ]:
feature_df["Is_Cancelled"] = (
    feature_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

print("Cancellation flag created successfully.")

## 10. Create Customer Type Feature

Customers are classified as:

- `Returning` if they appear in more than one invoice
- `New` if they appear in only one invoice
- `Unknown` if `CustomerID` is missing


In [ ]:
purchase_count = (
    feature_df.groupby("CustomerID")["InvoiceNo"]
    .transform("nunique")
)

feature_df["Customer_Type"] = np.where(
    feature_df["CustomerID"].isna(),
    "Unknown",
    np.where(purchase_count > 1, "Returning", "New")
)

print("Customer type feature created successfully.")

## 11. Create Invoice-Level Order Value Feature

`Average_Order_Value` stores the total invoice value against every row belonging to the same invoice.

This is useful for customer-level and invoice-level analysis.


In [ ]:
invoice_totals = (
    feature_df.groupby("InvoiceNo")["Total_Amount"]
    .transform("sum")
)

feature_df["Average_Order_Value"] = invoice_totals

print("Average order value feature created successfully.")

## 12. Create Region Feature

Countries are grouped into:

- `UK`
- `International`

This allows a simple comparison between domestic and international revenue.


In [ ]:
feature_df["Region"] = np.where(
    feature_df["Country"].eq("United Kingdom"),
    "UK",
    "International"
)

print("Region feature created successfully.")

## 13. Create Basic Product Category Feature

Product categories are derived from keywords in the `Description` column.

This is a basic rule-based category assignment and can be improved later with a more advanced product classification approach.


In [ ]:
# Standardize description values before applying keyword rules
feature_df["Description"] = (
    feature_df["Description"]
    .fillna("")
    .astype(str)
    .str.upper()
)

conditions = [
    feature_df["Description"].str.contains("HEART", na=False),
    feature_df["Description"].str.contains("BAG", na=False),
    feature_df["Description"].str.contains("CHRISTMAS", na=False),
    feature_df["Description"].str.contains("LAMP", na=False),
    feature_df["Description"].str.contains("BOX", na=False)
]

choices = [
    "Decor",
    "Bags",
    "Christmas",
    "Lighting",
    "Storage"
]

feature_df["Product_Category"] = np.select(
    conditions,
    choices,
    default="Other"
)

print("Product category feature created successfully.")

## 14. Create Weekend Indicator

`Is_Weekend` identifies whether the transaction happened on Saturday or Sunday.


In [ ]:
feature_df["Is_Weekend"] = feature_df["Weekday"].isin(["Saturday", "Sunday"])

print("Weekend indicator created successfully.")

## 15. Final Preview

The dataset should now contain the original columns plus engineered feature columns.


In [ ]:
feature_df.head()

## 16. Feature Engineering Summary

This summary records the new features created in this phase and their business purpose.


In [ ]:
feature_summary = pd.DataFrame({
    "Feature_Name": [
        "Invoice_Year",
        "Invoice_Month",
        "Month_Name",
        "Quarter",
        "Invoice_Day",
        "Weekday",
        "Hour",
        "Total_Amount",
        "Is_Cancelled",
        "Customer_Type",
        "Average_Order_Value",
        "Region",
        "Product_Category",
        "Is_Weekend"
    ],
    "Source_Column": [
        "InvoiceDate",
        "InvoiceDate",
        "InvoiceDate",
        "InvoiceDate",
        "InvoiceDate",
        "InvoiceDate",
        "InvoiceDate",
        "Quantity, UnitPrice",
        "InvoiceNo",
        "CustomerID, InvoiceNo",
        "InvoiceNo, Total_Amount",
        "Country",
        "Description",
        "Weekday"
    ],
    "Purpose": [
        "Year-wise sales analysis",
        "Monthly sales trend analysis",
        "Readable month labels for reporting",
        "Quarter-wise sales analysis",
        "Day-level transaction analysis",
        "Weekday purchase pattern analysis",
        "Hourly purchase behavior analysis",
        "Revenue calculation",
        "Cancellation identification",
        "New, returning, and unknown customer classification",
        "Invoice-level order value analysis",
        "UK versus international grouping",
        "Basic product grouping using description keywords",
        "Weekend transaction identification"
    ]
})

feature_summary

## 17. Save Outputs

The engineered dataset and summary report are saved into the project folder.


In [ ]:
# Save engineered dataset
feature_df.to_csv(feature_output_path, index=False)

# Save feature engineering summary report
feature_summary.to_csv(feature_summary_path, index=False)

print("Feature engineering outputs saved successfully.")
print(f"Engineered dataset: {feature_output_path}")
print(f"Summary report: {feature_summary_path}")

## 18. Final Validation

This final check confirms:

- Total rows after feature engineering
- Total columns after feature engineering
- Engineered columns added
- Missing values in engineered columns


In [ ]:
engineered_columns = [
    "Invoice_Year",
    "Invoice_Month",
    "Month_Name",
    "Quarter",
    "Invoice_Day",
    "Weekday",
    "Hour",
    "Total_Amount",
    "Is_Cancelled",
    "Customer_Type",
    "Average_Order_Value",
    "Region",
    "Product_Category",
    "Is_Weekend"
]

validation_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Feature Engineered Rows",
        "Original Columns",
        "Feature Engineered Columns",
        "New Features Created"
    ],
    "Value": [
        df.shape[0],
        feature_df.shape[0],
        df.shape[1],
        feature_df.shape[1],
        len(engineered_columns)
    ]
})

missing_summary = feature_df[engineered_columns].isna().sum().reset_index()
missing_summary.columns = ["Feature_Name", "Missing_Value_Count"]

print("Feature engineering validation summary:")
display(validation_summary)

print("Missing values in engineered columns:")
display(missing_summary)

# Phase 5 Completion Note

Phase 5 is complete after the following files are created:

```text
outputs/feature_engineered_online_retail.csv
reports/feature_engineering/feature_engineering_summary.csv
```

The dataset is now ready for Phase 6: Exploratory Data Analysis.
